# MPECSS - NosBench Benchmark (Group 2 of 3)

This notebook runs **Group 2** of the NosBench benchmark suite.

- **Dataset**: NosBench (Group 2)
- **Problems**: ~201 (balanced small/medium/large split)
- **Resume support**: built in via `kaggle_setup/resumable_benchmark.py`
- **Repo source**: GitHub (always clones fresh)

**Why split into groups?**
NosBench contains 600+ problems. Kaggle sessions have a 12-hour limit,
so we split the benchmark into 3 groups that can run in parallel on
separate Kaggle instances.

Run the cells in order. After a Kaggle restart, rerun the repository and
install cells, then run the **Resume** cell.

## 1. Configuration

In [ ]:
# ┌─────────────────────────────────────────────────┐
# │  NosBench Group 2 Configuration                  │
# └─────────────────────────────────────────────────┘
DATASET = 'nosbench'
GROUP = 2
RUN_TAG = f'Kaggle_NosBench_Group{GROUP}'

WORKERS = 4  # Match Kaggle's 4 CPU cores
TIMEOUT = 3600
NUM_PROBLEMS = None
PROBLEM_FILTER = ""
MEM_LIMIT_GB = None
SAVE_LOGS = True
SORT_BY_SIZE = False
SHUFFLE = True
RETRY_FAILED_ON_RESUME = False

# Data Source Detection
import os
KAGGLE_DATASET_PATH = '/kaggle/input/mpecss-benchmarks/benchmarks/nosbench/nosbench-json'
if not os.path.exists(KAGGLE_DATASET_PATH):
    KAGGLE_DATASET_PATH = None

# Repository source
REPO_DIR = "/kaggle/working/MPECSSCODEPAPER"
REPO_GIT_URL = "https://github.com/mrsaurabhtanwar/MPECSSCODEPAPER.git"

# Problem list for this group
PROBLEM_LIST = f"/kaggle/working/MPECSSCODEPAPER/kaggle_setup/nosbench_splits/nosbench_group{GROUP}_problems.txt"

## 2. Prepare The Repository

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_DIR = Path(REPO_DIR)

# Always clone fresh from GitHub to get latest code
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

print(f"Cloning repo from Git: {REPO_GIT_URL}")
subprocess.run(["git", "clone", REPO_GIT_URL, str(REPO_DIR)], check=True)

# Add to Python path
sys.path.insert(0, str(REPO_DIR))

print(f"Repository ready at: {REPO_DIR}")

## 3. Install Dependencies

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/MPECSSCODEPAPER

python -m pip install -q --upgrade pip
python -m pip install -q -e .

echo "[OK] Editable install completed"

## 4. Data Path Setup

In [ ]:
import os
DATASET_PATH = KAGGLE_DATASET_PATH if KAGGLE_DATASET_PATH else "/kaggle/working/MPECSSCODEPAPER/benchmarks/nosbench/nosbench-json"
print(f"Using benchmark directory: {DATASET_PATH}")
print(f"Using problem list: {PROBLEM_LIST}")

if not os.path.exists(DATASET_PATH):
    print(f"[ERROR] Benchmark path not found: {DATASET_PATH}")
if not os.path.exists(PROBLEM_LIST):
    print(f"[ERROR] Problem list not found: {PROBLEM_LIST}")
else:
    with open(PROBLEM_LIST) as f:
        count = sum(1 for line in f if line.strip() and not line.startswith('#'))
    print(f"Problems in this group: {count}")

## 5. Inspect The Kaggle Instance

In [ ]:
import multiprocessing
import platform
import psutil

print("=" * 60)
print("System Information")
print("=" * 60)
print(f"Platform: {platform.platform()}")
print(f"Python: {platform.python_version()}")
print(f"Logical CPU cores: {multiprocessing.cpu_count()}")

mem = psutil.virtual_memory()
print(f"Total RAM: {mem.total / 1024**3:.1f} GB")
print(f"Available RAM: {mem.available / 1024**3:.1f} GB")
print("=" * 60)

## 6. Run Preflight Checks

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/MPECSSCODEPAPER
python scripts/preflight_checks.py

## 7. Load The Runner Helper

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path(REPO_DIR)
WRAPPER = REPO_DIR / "kaggle_setup" / "resumable_benchmark.py"

def run_benchmark(resume_latest=False, summary_only=False):
    command = [
        sys.executable,
        str(WRAPPER),
        "--dataset", DATASET,
        "--repo-dir", str(REPO_DIR),
        "--tag", RUN_TAG,
        "--workers", str(WORKERS),
        "--timeout", str(TIMEOUT),
        "--skip-preflight",
    ]

    if SAVE_LOGS:
        command.append("--save-logs")
    if SORT_BY_SIZE:
        command.append("--sort-by-size")
    if SHUFFLE:
        command.append("--shuffle")
    else:
        command.append("--no-shuffle")
    if PROBLEM_FILTER:
        command.extend(["--problem", PROBLEM_FILTER])
    if NUM_PROBLEMS is not None:
        command.extend(["--num-problems", str(NUM_PROBLEMS)])
    if MEM_LIMIT_GB is not None:
        command.extend(["--mem-limit-gb", str(MEM_LIMIT_GB)])
    if DATASET_PATH:
        command.extend(["--path", str(DATASET_PATH)])
    if PROBLEM_LIST:
        command.extend(["--problem-list", str(PROBLEM_LIST)])
    if RETRY_FAILED_ON_RESUME:
        command.append("--retry-failed")
    if resume_latest:
        command.append("--resume-latest")
    if summary_only:
        command.append("--summary-only")

    print("+", " ".join(command))
    subprocess.run(command, check=True)

## 8. Launch A Fresh Run

In [ ]:
run_benchmark()

## 9. Resume After A Kaggle Restart

In [ ]:
run_benchmark(resume_latest=True)

## 10. Progress Summary

In [ ]:
run_benchmark(summary_only=True)

## 11. Copy Results For Download

In [ ]:
%%bash
set -euo pipefail
mkdir -p /kaggle/working/outputs
cp -r /kaggle/working/MPECSSCODEPAPER/results/* /kaggle/working/outputs/ || true

echo "[OK] Results copied to /kaggle/working/outputs/"
echo "Download from the Kaggle file browser or output panel"

## 12. Software Versions For Your Paper

In [ ]:
import casadi
import matplotlib
import numpy
import pandas
import platform
import psutil
import scipy

print("Software Versions")
print("=" * 40)
print(f"Python: {platform.python_version()}")
print(f"NumPy: {numpy.__version__}")
print(f"SciPy: {scipy.__version__}")
print(f"Pandas: {pandas.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"CasADi: {casadi.__version__}")
print(f"psutil: {psutil.__version__}")
print("=" * 40)